# last_test — Loss / Feature subset / Target transform 다양화 stacking 실험

**목적**: 11-base stacking plateau (val=0.005701) 깰 수 있는 신규 다양성 base를 만들고 게이트(잔차 corr<0.97)로 거른 뒤 stacking 개선 여부 확인.

**기존 search space와 중복 안 되는 영역만 선별**:

| 차원 | 새 영역 | 기존 탐색 (드롭됨) |
|---|---|---|
| Loss | Tweedie 1.7 / 1.8 | regression / poisson / tweedie 1.2 / 1.5 |
| Target transform | none + tweedie | log1p (고정) |
| Feature subset | top_200 (gain) | full 568만 |
| ZIT zeta | 1.10~1.90 모두 탐색됨 → **드롭** | 180 trials |

**구성**:
- Study A: reg_lgbm 변형 (GridSampler, 4 trials) — `objective` × `target_transform`
- Study B: two_stage_reverse 변형 (GridSampler, 6→5 trials) — `reg_objective` × `feat_subset`
- 게이트: 단독 val < 0.005994 AND max_corr_with_11base < 0.97
- Stacking 비교: 11-base only / 11-base + 통과 후보

**예상 시간**: 데이터 로드 15분 + Study A 30분 + Study B 75분 + 게이트/스태킹 30분 = ~2.5시간

## 1. 환경 + import (Colab GPU / Local 공통)

In [1]:
import os, sys, json

GDRIVE_FINAL_ID = '1HR7LlQmp4n9wGh2WneyVex2mCZ-poiY9'

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system('gdown 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/final/modules/preprocess.py'):
        assert GDRIVE_FINAL_ID, 'GDRIVE_FINAL_ID 비어있음'
        os.makedirs('/content/project/3_modeling/final', exist_ok=True)
        os.system(f'gdown {GDRIVE_FINAL_ID} -O /content/final.zip')
        os.system('unzip -qo /content/final.zip -d /content/project/3_modeling/final')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess

import lightgbm as lgb
import optuna
from optuna.samplers import GridSampler
from sklearn.model_selection import KFold
from sklearn.linear_model import ElasticNetCV

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'SEED = {SEED}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트
SEED = 42


## 2. 설정 + 게이트 임계값

In [2]:
USER       = 'jh'
N_FOLDS    = 5
CLIP_Y_EXTREME = True

# ── 게이트 임계값 ──
BEST_SINGLE_VAL = 0.005709     # 11-base 단독 best (zit_only/bag_zit_combined_best)
THR_SINGLE_VAL  = BEST_SINGLE_VAL * 1.05   # 0.005994
THR_CORR_VS_11  = 0.97
THR_CORR_PEER   = 0.95

# ── 출력 경로 ──
OUT_DIR = os.path.join(OUTPUT_DIR, 'last_test')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, 'reg_lgbm'), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, 'two_stage_reverse'), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, 'stacking'), exist_ok=True)

# ── 베이스 HP 출처 (각 architecture 별 최고 파라미터) ──
LGBM_BEST_HP_PATH = os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'lgbm', 'best_params.json')
TS_REVERSE_BEST_PATH = os.path.join(OUTPUT_DIR, '_temp', 'two_stage_reverse', 'best_params.json')

# ── 11-base OOF 출처 (게이트 잔차 corr 계산용) ──
BASE_OOF_PATHS = {
    'zit_only':                 os.path.join(OUTPUT_DIR, 'final', 'zit_only',      'oof_unit.csv'),
    'bag_zit_combined_best':    os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_combined_best',    'oof_unit.csv'),
    'bag_zit_hpo':              os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_hpo',              'oof_unit.csv'),
    'bag_zit_combined_best_xy': os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_combined_best_xy', 'oof_unit.csv'),
    'bag_zit_pp_hpo':           os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_pp_hpo',           'oof_unit.csv'),
    'bag_zit_fixed_ge':         os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_fixed_ge',         'oof_unit.csv'),
    'reg__catboost':            os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'catboost', 'oof_unit.csv'),
    'reg__lgbm':                os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'lgbm',     'oof_unit.csv'),
    'reg__et':                  os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'et',       'oof_unit.csv'),
    'reg__enet':                os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'enet',     'oof_unit.csv'),
}
BASE_VAL_PATHS = {k: v.replace('oof_unit', 'val_unit')  for k, v in BASE_OOF_PATHS.items()}
BASE_TEST_PATHS = {k: v.replace('oof_unit', 'test_unit') for k, v in BASE_OOF_PATHS.items()}

# ── 전처리 PARAMS (03b log1p preset — reg_lgbm/two_stage_reverse 둘 다 동일) ──
PARAMS = {
    'missing_threshold':          0.5,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.25,
    'spatial_max_dist':           5.0,
    'post_impute_corr_threshold': 0.99,
    'post_impute_corr_keep_by':   'std',
}

print(f'OUT_DIR={OUT_DIR}')
print(f'게이트 임계: single_val<{THR_SINGLE_VAL:.6f}, corr_vs_11base<{THR_CORR_VS_11}, peer_corr<{THR_CORR_PEER}')
print(f'N_FOLDS={N_FOLDS}, CLIP_Y_EXTREME={CLIP_Y_EXTREME}')

OUT_DIR=c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\last_test
게이트 임계: single_val<0.005994, corr_vs_11base<0.97, peer_corr<0.95
N_FOLDS=5, CLIP_Y_EXTREME=True


## 3. 데이터 로드 + 전처리 (1회만)

`reg_lgbm`과 `two_stage_reverse`는 동일 03b log1p preset 사용 → 1번만 로드/전처리하고 두 study에서 공유.

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = int((y_raw >= 1.0).sum())
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip ({n_clipped}개)')

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

# 전처리 (1회)
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

X_train_die = xs_train_die[feat_cols_clean].values.astype(np.float64)
X_val_die   = xs_val_die[feat_cols_clean].values.astype(np.float64)
X_test_die  = xs_test_die[feat_cols_clean].values.astype(np.float64)
uid_train_die = xs_train_die[KEY_COL].values
uid_val_die   = xs_val_die[KEY_COL].values
uid_test_die  = xs_test_die[KEY_COL].values

# unit broadcast
y_train_die_broadcast = pd.Series(uid_train_die).map(y_train_unit).values.astype(np.float64)
assert not pd.isna(y_train_die_broadcast).any(), 'unmapped train die y'
y_bin_die_broadcast = (y_train_die_broadcast > 0).astype(np.int32)

# unit-level KFold
unit_ids_train_unique = y_train_unit.index.values
kf_global = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf_global.split(unit_ids_train_unique))

n_train_die = len(X_train_die)
n_val_die   = len(X_val_die)
n_test_die  = len(X_test_die)

print(f'\n[cleaning] feat_cols_clean={len(feat_cols_clean)}')
print(f'  X_train_die: {X_train_die.shape}, val: {X_val_die.shape}, test: {X_test_die.shape}')
print(f'  unit train={len(y_train_unit):,}, val={len(y_val_unit):,}, test={len(y_test_unit):,}')
print(f'  fold: {N_FOLDS}-fold unit-level shuffle SEED={SEED}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip (1개)
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=50%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 332개, 잔여: 564개
    컬럼: 896 → 564 (332개 제거)
    DataFrame: (104748, 622)

[결측 indicator] 4개 컬럼 추가 (결측률 >= 25%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, di

## 4. 11-base OOF/val/test 잔차 로드

각 base의 `oof_unit.csv`에서 `pred` 컬럼만 가져와 잔차(`y - pred`) 계산. 게이트 corr 검사용.

In [4]:
def _load_base_pred(path, y_series):
    df = pd.read_csv(path)
    return df.set_index(KEY_COL)['pred'].reindex(y_series.index).values

base_oof_pred  = {k: _load_base_pred(p, y_train_unit) for k, p in BASE_OOF_PATHS.items()}
base_val_pred  = {k: _load_base_pred(p, y_val_unit)   for k, p in BASE_VAL_PATHS.items()}
base_test_pred = {k: _load_base_pred(p, y_test_unit)  for k, p in BASE_TEST_PATHS.items()}

base_oof_resid  = {k: y_train_unit.values - v for k, v in base_oof_pred.items()}

print(f'[11-base 로드] {len(base_oof_pred)}개')
for k in base_oof_pred:
    p = base_oof_pred[k]
    rmse = float(np.sqrt(np.mean((y_train_unit.values - p) ** 2)))
    print(f'  {k:30s}  oof_rmse={rmse:.6f}')

[11-base 로드] 10개
  zit_only                        oof_rmse=0.005503
  bag_zit_combined_best           oof_rmse=0.005507
  bag_zit_hpo                     oof_rmse=0.005501
  bag_zit_combined_best_xy        oof_rmse=0.005509
  bag_zit_pp_hpo                  oof_rmse=0.005500
  bag_zit_fixed_ge                oof_rmse=0.005514
  reg__catboost                   oof_rmse=0.005523
  reg__lgbm                       oof_rmse=0.005520
  reg__et                         oof_rmse=0.005542
  reg__enet                       oof_rmse=0.005563


## 5. Feature gain importance (top_200 인덱스 캐싱)

`final/reg_only/lgbm/fold_models.pkl`의 5-fold 평균 gain importance로 top_200 feature 선택. two_stage_reverse `feat_subset='top_200'` 변형에서 사용.

In [5]:
import pickle

with open(os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'lgbm', 'fold_models.pkl'), 'rb') as f:
    fm_data = pickle.load(f)

# fold_models.pkl 구조: {'fold_models': [LGBMRegressor × 5], 'feature_names': [...], ...}
fold_models_list = fm_data['fold_models']
lgbm_feat_names  = fm_data['feature_names']
print(f'fold_models 구조: dict {list(fm_data.keys())}')
print(f'  fold_models: {len(fold_models_list)}개 LGBMRegressor')
print(f'  feature_names: {len(lgbm_feat_names)}개')

# 5-fold 평균 gain importance
gains = []
for m in fold_models_list:
    if hasattr(m, 'booster_'):
        gains.append(m.booster_.feature_importance(importance_type='gain'))
    elif hasattr(m, 'feature_importances_'):
        gains.append(m.feature_importances_)
mean_gain = np.mean(gains, axis=0)
assert len(mean_gain) == len(lgbm_feat_names), f'gain {len(mean_gain)} vs feat_names {len(lgbm_feat_names)}'

# feat_cols_clean과의 교집합 → top_200
common_feats = [f for f in lgbm_feat_names if f in feat_cols_clean]
print(f'  교집합 (lgbm_feat_names ∩ feat_cols_clean): {len(common_feats)}개')

gain_dict = {f: g for f, g in zip(lgbm_feat_names, mean_gain)}
common_gains = np.array([gain_dict[f] for f in common_feats])
top_200_feats = [f for f, g in sorted(zip(common_feats, common_gains), key=lambda x: -x[1])[:200]]
top_200_idx_in_clean = [feat_cols_clean.index(f) for f in top_200_feats]

print(f'top_200 features 선택 완료. 첫 5: {top_200_feats[:5]}')
top_200_gain_sum = sum(common_gains[np.argsort(-common_gains)[:200]])
print(f'  cumulative gain ratio (top_200 / total): {top_200_gain_sum / common_gains.sum():.3f}')

fold_models 구조: dict ['fold_models', 'fold_scalers', 'feature_names', 'extra_feature_name', 'model_name', 'n_folds']
  fold_models: 5개 LGBMRegressor
  feature_names: 568개
  교집합 (lgbm_feat_names ∩ feat_cols_clean): 568개
top_200 features 선택 완료. 첫 5: ['X1049', 'X1066', 'X592', 'X727', 'X1060']
  cumulative gain ratio (top_200 / total): 0.831


## 6. 헬퍼: 변형 학습 + 저장

두 architecture 모두 5-fold CV 학습 + die-level 예측 → unit-level 평균 집계 → oof/val/test_unit.csv + meta.json 저장.

In [6]:
import time

def _mean_die_to_unit(pred_die, uid_die):
    unit_id = np.asarray(uid_die)
    unique_units, inverse = np.unique(unit_id, return_inverse=True)
    pred_sum = np.zeros(len(unique_units))
    cnt      = np.zeros(len(unique_units))
    np.add.at(pred_sum, inverse, pred_die)
    np.add.at(cnt,      inverse, 1.0)
    return pred_sum / cnt, unique_units


def _rmse_unit(pred_unit_arr, unique_units, y_unit_series):
    s = pd.Series(pred_unit_arr, index=unique_units).reindex(y_unit_series.index)
    return float(np.sqrt(np.mean((s.values - y_unit_series.values) ** 2)))


def _save_unit_csv(uids, pred, y_true, path):
    pd.DataFrame({KEY_COL: uids, 'pred': pred, TARGET_COL: y_true}).to_csv(path, index=False)


# ─────────────────────────────────────────────────────────────
# Variant A: reg_lgbm (objective × target_transform 변형)
# ─────────────────────────────────────────────────────────────

# reg_lgbm 베이스 HP
with open(LGBM_BEST_HP_PATH) as f:
    LGBM_BASE_HP_FULL = json.load(f)['best_params_resolved']

def train_reg_lgbm_variant(objective_str, target_transform, save_dir):
    """
    objective_str: 'tweedie_1.7' | 'tweedie_1.8' (tweedie_X.X)
    target_transform: 'log1p' | 'none'
    """
    os.makedirs(save_dir, exist_ok=True)
    hp = dict(LGBM_BASE_HP_FULL)
    # objective 오버라이드
    hp.pop('objective', None)
    hp.pop('tweedie_variance_power', None)
    if objective_str.startswith('tweedie'):
        hp['objective'] = 'tweedie'
        hp['tweedie_variance_power'] = float(objective_str.split('_')[1])
    else:
        hp['objective'] = objective_str
    # 안전 고정
    hp['random_state'] = SEED
    hp['n_jobs'] = -1
    hp['verbose'] = -1
    hp.setdefault('subsample_freq', 1)

    oof_die_pred  = np.full(n_train_die, np.nan)
    val_die_pred  = np.zeros(n_val_die)
    test_die_pred = np.zeros(n_test_die)

    t0 = time.time()
    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unit_ids_train_unique[tr_uidx]
        vl_units = unit_ids_train_unique[vl_uidx]
        tr_die_mask = np.isin(uid_train_die, tr_units)
        vl_die_mask = np.isin(uid_train_die, vl_units)
        X_tr = X_train_die[tr_die_mask]
        X_vl = X_train_die[vl_die_mask]
        y_tr = y_train_die_broadcast[tr_die_mask]

        if target_transform == 'log1p':
            y_tr_fit = np.log1p(y_tr)
        elif target_transform == 'none':
            y_tr_fit = y_tr
        else:
            raise ValueError(f'unknown target_transform: {target_transform}')

        m = lgb.LGBMRegressor(**hp)
        m.fit(X_tr, y_tr_fit)

        for X_pred, target_arr in [
            (X_vl,       oof_die_pred),
            (X_val_die,  val_die_pred),
            (X_test_die, test_die_pred),
        ]:
            p = m.predict(X_pred)
            if target_transform == 'log1p':
                p = np.clip(np.expm1(p), 0.0, None)
            else:
                p = np.clip(p, 0.0, None)
            if X_pred is X_vl:
                target_arr[vl_die_mask] = p
            else:
                target_arr += p / N_FOLDS

    assert not np.isnan(oof_die_pred).any()

    # die→unit 집계
    oof_u, oof_uids   = _mean_die_to_unit(oof_die_pred,  uid_train_die)
    val_u, val_uids   = _mean_die_to_unit(val_die_pred,  uid_val_die)
    test_u, test_uids = _mean_die_to_unit(test_die_pred, uid_test_die)

    oof_rmse  = _rmse_unit(oof_u,  oof_uids,  y_train_unit)
    val_rmse  = _rmse_unit(val_u,  val_uids,  y_val_unit)
    test_rmse = _rmse_unit(test_u, test_uids, y_test_unit)

    # 정렬 후 저장
    oof_s  = pd.Series(oof_u,  index=oof_uids).reindex(y_train_unit.index)
    val_s  = pd.Series(val_u,  index=val_uids).reindex(y_val_unit.index)
    test_s = pd.Series(test_u, index=test_uids).reindex(y_test_unit.index)
    _save_unit_csv(y_train_unit.index.values, oof_s.values,  y_train_unit.values, os.path.join(save_dir, 'oof_unit.csv'))
    _save_unit_csv(y_val_unit.index.values,   val_s.values,  y_val_unit.values,   os.path.join(save_dir, 'val_unit.csv'))
    _save_unit_csv(y_test_unit.index.values,  test_s.values, y_test_unit.values,  os.path.join(save_dir, 'test_unit.csv'))

    meta = {
        'architecture': 'reg_lgbm',
        'variant_objective': objective_str,
        'variant_target_transform': target_transform,
        'training_level': 'die-level broadcast',
        'die_to_unit_agg': 'mean',
        'n_folds': N_FOLDS,
        'oof_rmse': oof_rmse,
        'val_rmse': val_rmse,
        'test_rmse': test_rmse,
        'preprocess_PARAMS': PARAMS,
        'effective_pp_params': pp['effective_params'],
        'best_hp_resolved': hp,
        'CLIP_Y_EXTREME': CLIP_Y_EXTREME,
        'feat_cols_clean_n': len(feat_cols_clean),
        'SEED': int(SEED),
        'elapsed_seconds': time.time() - t0,
    }
    with open(os.path.join(save_dir, 'meta.json'), 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

    return {'oof_rmse': oof_rmse, 'val_rmse': val_rmse, 'test_rmse': test_rmse,
            'oof_pred': oof_s.values, 'val_pred': val_s.values, 'test_pred': test_s.values,
            'elapsed': time.time() - t0}


# ─────────────────────────────────────────────────────────────
# Variant B: two_stage_reverse (reg_objective × feat_subset 변형)
# ─────────────────────────────────────────────────────────────

# two_stage_reverse 베이스 HP
with open(TS_REVERSE_BEST_PATH) as f:
    TS_REVERSE_BEST = json.load(f)
TS_HP_BEST = TS_REVERSE_BEST['hp_best']
TS_W0      = TS_REVERSE_BEST['best_w0']
TS_CLF_SPW = TS_REVERSE_BEST['best_clf_spw']

def _train_path_b_one_fold(X_tr, y_tr_continuous, y_tr_bin, X_others, hp, w0, reg_obj, clf_spw):
    """path B 1 fold (two_stage_reverse_hpo.ipynb의 _train_path_b 그대로)."""
    sw = np.where(y_tr_continuous == 0, w0, 1.0)
    y_tr_log = np.log1p(y_tr_continuous)
    reg_params = dict(hp)
    if reg_obj.startswith('tweedie'):
        reg_params['objective'] = 'tweedie'
        reg_params['tweedie_variance_power'] = float(reg_obj.split('_')[1])
    else:
        reg_params['objective'] = reg_obj
    reg = lgb.LGBMRegressor(**reg_params)
    reg.fit(X_tr, y_tr_log, sample_weight=sw)

    reg_train_log = reg.predict(X_tr)
    reg_train_y   = np.clip(np.expm1(reg_train_log), 0.0, None)
    X_tr_aug = np.hstack([X_tr, reg_train_y.reshape(-1, 1)])

    clf_params = dict(hp)
    clf_params['objective'] = 'binary'
    clf_params['scale_pos_weight'] = clf_spw
    clf = lgb.LGBMClassifier(**clf_params)
    clf.fit(X_tr_aug, y_tr_bin)

    results = []
    for X_o in X_others:
        reg_log_o = reg.predict(X_o)
        reg_y_o   = np.clip(np.expm1(reg_log_o), 0.0, None)
        X_o_aug   = np.hstack([X_o, reg_y_o.reshape(-1, 1)])
        prob_o = clf.predict_proba(X_o_aug)[:, 1]
        prob_o = np.clip(prob_o, 0.0, 1.0)
        final_o = prob_o * reg_y_o
        results.append(final_o)
    return results


def train_ts_reverse_variant(reg_objective, feat_subset, save_dir):
    """
    reg_objective: 'regression' | 'tweedie_1.7' | 'tweedie_1.8'
    feat_subset:   'full' | 'top_200'
    """
    os.makedirs(save_dir, exist_ok=True)
    hp = dict(TS_HP_BEST)
    hp['random_state'] = SEED
    hp['n_jobs'] = -1
    hp['verbose'] = -1
    hp.setdefault('subsample_freq', 1)

    if feat_subset == 'top_200':
        col_idx = top_200_idx_in_clean
        X_tr_full  = X_train_die[:, col_idx]
        X_vl_full  = X_val_die[:, col_idx]
        X_te_full  = X_test_die[:, col_idx]
    elif feat_subset == 'full':
        X_tr_full  = X_train_die
        X_vl_full  = X_val_die
        X_te_full  = X_test_die
    else:
        raise ValueError(f'unknown feat_subset: {feat_subset}')

    oof_die_pred  = np.full(n_train_die, np.nan)
    val_die_pred  = np.zeros(n_val_die)
    test_die_pred = np.zeros(n_test_die)

    t0 = time.time()
    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unit_ids_train_unique[tr_uidx]
        vl_units = unit_ids_train_unique[vl_uidx]
        tr_die_mask = np.isin(uid_train_die, tr_units)
        vl_die_mask = np.isin(uid_train_die, vl_units)
        X_tr  = X_tr_full[tr_die_mask]
        X_vl  = X_tr_full[vl_die_mask]
        y_tr  = y_train_die_broadcast[tr_die_mask]
        yb_tr = y_bin_die_broadcast[tr_die_mask]

        f_vl, f_v, f_t = _train_path_b_one_fold(
            X_tr, y_tr, yb_tr,
            [X_vl, X_vl_full, X_te_full],
            hp, TS_W0, reg_objective, TS_CLF_SPW,
        )
        oof_die_pred[vl_die_mask] = f_vl
        val_die_pred  += f_v / N_FOLDS
        test_die_pred += f_t / N_FOLDS

    assert not np.isnan(oof_die_pred).any()

    oof_u, oof_uids   = _mean_die_to_unit(oof_die_pred,  uid_train_die)
    val_u, val_uids   = _mean_die_to_unit(val_die_pred,  uid_val_die)
    test_u, test_uids = _mean_die_to_unit(test_die_pred, uid_test_die)

    oof_rmse  = _rmse_unit(oof_u,  oof_uids,  y_train_unit)
    val_rmse  = _rmse_unit(val_u,  val_uids,  y_val_unit)
    test_rmse = _rmse_unit(test_u, test_uids, y_test_unit)

    oof_s  = pd.Series(oof_u,  index=oof_uids).reindex(y_train_unit.index)
    val_s  = pd.Series(val_u,  index=val_uids).reindex(y_val_unit.index)
    test_s = pd.Series(test_u, index=test_uids).reindex(y_test_unit.index)
    _save_unit_csv(y_train_unit.index.values, oof_s.values,  y_train_unit.values, os.path.join(save_dir, 'oof_unit.csv'))
    _save_unit_csv(y_val_unit.index.values,   val_s.values,  y_val_unit.values,   os.path.join(save_dir, 'val_unit.csv'))
    _save_unit_csv(y_test_unit.index.values,  test_s.values, y_test_unit.values,  os.path.join(save_dir, 'test_unit.csv'))

    meta = {
        'architecture': 'two_stage_reverse',
        'variant_reg_objective': reg_objective,
        'variant_feat_subset': feat_subset,
        'n_features_used': X_tr_full.shape[1],
        'training_level': 'die-level broadcast',
        'die_to_unit_agg': 'mean',
        'n_folds': N_FOLDS,
        'oof_rmse': oof_rmse,
        'val_rmse': val_rmse,
        'test_rmse': test_rmse,
        'preprocess_PARAMS': PARAMS,
        'effective_pp_params': pp['effective_params'],
        'hp_best': hp,
        'best_w0': TS_W0,
        'best_clf_spw': TS_CLF_SPW,
        'CLIP_Y_EXTREME': CLIP_Y_EXTREME,
        'feat_cols_clean_n': len(feat_cols_clean),
        'SEED': int(SEED),
        'elapsed_seconds': time.time() - t0,
    }
    with open(os.path.join(save_dir, 'meta.json'), 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

    return {'oof_rmse': oof_rmse, 'val_rmse': val_rmse, 'test_rmse': test_rmse,
            'oof_pred': oof_s.values, 'val_pred': val_s.values, 'test_pred': test_s.values,
            'elapsed': time.time() - t0}


print('헬퍼 정의 완료')

헬퍼 정의 완료


## 7. Study A — reg_lgbm 변형 (GridSampler, 4 trials)

`objective` × `target_transform` = 2 × 2 = 4 cells. 모두 새 영역 (기존 reg_lgbm HPO에는 tweedie 1.7/1.8 없음, target_transform='none'+tweedie 조합도 없음).

In [7]:
RUN_STUDY_A = True
study_a_records = []   # 모든 trial 결과 (게이트용)

if RUN_STUDY_A:
    search_space_a = {
        'objective':        ['tweedie_1.7', 'tweedie_1.8'],
        'target_transform': ['log1p', 'none'],
    }
    study_a_id = 'last-test-reg-lgbm'
    study_a_db = os.path.join(OUT_DIR, 'reg_lgbm', f'optuna_{USER}_{study_a_id}.db')
    if os.path.exists(study_a_db):
        os.remove(study_a_db)

    def objective_a(trial):
        obj_choice = trial.suggest_categorical('objective', search_space_a['objective'])
        tt         = trial.suggest_categorical('target_transform', search_space_a['target_transform'])
        cell_name = f'{obj_choice}__{tt}'
        save_dir = os.path.join(OUT_DIR, 'reg_lgbm', f'trial_{trial.number}_{cell_name}')
        print(f'\n--- Study A trial {trial.number}: {cell_name} ---')
        r = train_reg_lgbm_variant(obj_choice, tt, save_dir)
        trial.set_user_attr('cell_name', cell_name)
        trial.set_user_attr('val_rmse', r['val_rmse'])
        trial.set_user_attr('test_rmse', r['test_rmse'])
        trial.set_user_attr('save_dir', save_dir)
        study_a_records.append({
            'study': 'A_reg_lgbm', 'trial': trial.number, 'cell_name': cell_name,
            'oof_rmse': r['oof_rmse'], 'val_rmse': r['val_rmse'], 'test_rmse': r['test_rmse'],
            'save_dir': save_dir, 'oof_pred': r['oof_pred'], 'val_pred': r['val_pred'], 'test_pred': r['test_pred'],
        })
        print(f'  oof={r["oof_rmse"]:.6f}  val={r["val_rmse"]:.6f}  test={r["test_rmse"]:.6f}  ({r["elapsed"]:.0f}s)')
        return r['oof_rmse']

    sampler_a = GridSampler(search_space_a)
    study_a = optuna.create_study(
        direction='minimize',
        study_name=study_a_id,
        storage=f'sqlite:///{study_a_db}',
        sampler=sampler_a,
        load_if_exists=False,
    )
    n_trials_a = len(search_space_a['objective']) * len(search_space_a['target_transform'])
    print(f'=== Study A 시작: {n_trials_a} trials (GridSampler) ===')
    t_total = time.time()
    study_a.optimize(objective_a, n_trials=n_trials_a)
    print(f'\n[Study A 완료] {time.time()-t_total:.0f}s, {len(study_a_records)} trials')
else:
    print('RUN_STUDY_A=False → 스킵')

=== Study A 시작: 4 trials (GridSampler) ===

--- Study A trial 0: tweedie_1.8__log1p ---
  oof=0.005540  val=0.005750  test=0.008445  (311s)

--- Study A trial 1: tweedie_1.8__none ---
  oof=0.005539  val=0.005750  test=0.008444  (230s)

--- Study A trial 2: tweedie_1.7__none ---
  oof=0.005534  val=0.005745  test=0.008440  (231s)

--- Study A trial 3: tweedie_1.7__log1p ---
  oof=0.005535  val=0.005745  test=0.008441  (235s)

[Study A 완료] 1008s, 4 trials


## 8. Study B — two_stage_reverse 변형 (GridSampler, 6 cells, 1 pruned)

`reg_objective` × `feat_subset` = 3 × 2 = 6 cells. `(regression, full)`은 기존 best와 정확히 동일 → trial 안에서 pruned로 스킵 → **5 cell 실행**.

In [8]:
RUN_STUDY_B = True
study_b_records = []

if RUN_STUDY_B:
    search_space_b = {
        'reg_objective': ['regression', 'tweedie_1.7', 'tweedie_1.8'],
        'feat_subset':   ['full', 'top_200'],
    }
    study_b_id = 'last-test-ts-reverse'
    study_b_db = os.path.join(OUT_DIR, 'two_stage_reverse', f'optuna_{USER}_{study_b_id}.db')
    if os.path.exists(study_b_db):
        os.remove(study_b_db)

    def objective_b(trial):
        reg_obj = trial.suggest_categorical('reg_objective', search_space_b['reg_objective'])
        feat    = trial.suggest_categorical('feat_subset',   search_space_b['feat_subset'])
        cell_name = f'{reg_obj}__{feat}'
        # 기존 best (regression+full)는 스킵
        if (reg_obj == 'regression') and (feat == 'full'):
            print(f'\n--- Study B trial {trial.number}: {cell_name} → SKIP (기존 best와 동일) ---')
            raise optuna.TrialPruned()
        save_dir = os.path.join(OUT_DIR, 'two_stage_reverse', f'trial_{trial.number}_{cell_name}')
        print(f'\n--- Study B trial {trial.number}: {cell_name} ---')
        r = train_ts_reverse_variant(reg_obj, feat, save_dir)
        trial.set_user_attr('cell_name', cell_name)
        trial.set_user_attr('val_rmse', r['val_rmse'])
        trial.set_user_attr('test_rmse', r['test_rmse'])
        trial.set_user_attr('save_dir', save_dir)
        study_b_records.append({
            'study': 'B_two_stage_reverse', 'trial': trial.number, 'cell_name': cell_name,
            'oof_rmse': r['oof_rmse'], 'val_rmse': r['val_rmse'], 'test_rmse': r['test_rmse'],
            'save_dir': save_dir, 'oof_pred': r['oof_pred'], 'val_pred': r['val_pred'], 'test_pred': r['test_pred'],
        })
        print(f'  oof={r["oof_rmse"]:.6f}  val={r["val_rmse"]:.6f}  test={r["test_rmse"]:.6f}  ({r["elapsed"]:.0f}s)')
        return r['oof_rmse']

    sampler_b = GridSampler(search_space_b)
    study_b = optuna.create_study(
        direction='minimize',
        study_name=study_b_id,
        storage=f'sqlite:///{study_b_db}',
        sampler=sampler_b,
        load_if_exists=False,
    )
    n_trials_b = len(search_space_b['reg_objective']) * len(search_space_b['feat_subset'])
    print(f'=== Study B 시작: {n_trials_b} trials → 1 pruned 예정 ({n_trials_b-1} 실행) ===')
    t_total = time.time()
    study_b.optimize(objective_b, n_trials=n_trials_b)
    print(f'\n[Study B 완료] {time.time()-t_total:.0f}s, {len(study_b_records)} trials 실행')
else:
    print('RUN_STUDY_B=False → 스킵')

=== Study B 시작: 6 trials → 1 pruned 예정 (5 실행) ===

--- Study B trial 0: tweedie_1.8__top_200 ---
  oof=0.005616  val=0.005815  test=0.008487  (249s)

--- Study B trial 1: tweedie_1.8__full ---
  oof=0.005624  val=0.005823  test=0.008494  (345s)

--- Study B trial 2: tweedie_1.7__full ---
  oof=0.005598  val=0.005799  test=0.008478  (348s)

--- Study B trial 3: regression__top_200 ---
  oof=0.005497  val=0.005707  test=0.008412  (207s)

--- Study B trial 4: regression__full → SKIP (기존 best와 동일) ---

--- Study B trial 5: tweedie_1.7__top_200 ---
  oof=0.005596  val=0.005795  test=0.008473  (262s)

[Study B 완료] 1411s, 5 trials 실행


## 9. 게이트 검사

각 신규 trial에 대해:
1. **단독 val RMSE < 0.005994** (= 1.05 × 0.005709)
2. **11-base와 OOF 잔차 max corr < 0.97**
3. **통과 후보들 사이 mutual corr < 0.95**

세 조건 모두 통과하면 stacking pool 추가 후보.

In [9]:
all_records = study_a_records + study_b_records
print(f'전체 신규 trial: {len(all_records)}')

# 게이트 1+2
gate_rows = []
for r in all_records:
    new_resid = y_train_unit.values - r['oof_pred']
    corr_with_11 = {
        bn: float(np.corrcoef(new_resid, base_oof_resid[bn])[0, 1])
        for bn in base_oof_resid
    }
    max_corr = max(corr_with_11.values())
    argmax_corr = max(corr_with_11, key=corr_with_11.get)
    pass_1 = r['val_rmse'] < THR_SINGLE_VAL
    pass_2 = max_corr < THR_CORR_VS_11
    gate_rows.append({
        **{k: v for k, v in r.items() if k not in ('oof_pred', 'val_pred', 'test_pred')},
        'max_corr_vs_11': max_corr,
        'argmax_corr':    argmax_corr,
        'pass_1':         pass_1,
        'pass_2':         pass_2,
        'pre_pass':       pass_1 and pass_2,
    })

gate_df = pd.DataFrame(gate_rows).sort_values('val_rmse')
print('\n=== 게이트 1+2 검사 결과 ===')
print(gate_df[['study','trial','cell_name','val_rmse','max_corr_vs_11','argmax_corr','pass_1','pass_2','pre_pass']].to_string(index=False))

pre_passed = [r for r in all_records if any(g['trial']==r['trial'] and g['study']==r['study'] and g['pre_pass'] for g in gate_rows)]
print(f'\n게이트 1+2 통과: {len(pre_passed)}/{len(all_records)}')

# 게이트 3: peer mutual corr (통과 후보들 사이)
final_passed = []
if len(pre_passed) >= 2:
    pre_resid = [(r, y_train_unit.values - r['oof_pred']) for r in pre_passed]
    keep = list(range(len(pre_resid)))
    drop = set()
    for i in range(len(pre_resid)):
        if i in drop: continue
        for j in range(i+1, len(pre_resid)):
            if j in drop: continue
            c = float(np.corrcoef(pre_resid[i][1], pre_resid[j][1])[0, 1])
            if c >= THR_CORR_PEER:
                # 단독 val 더 큰 쪽 drop
                if pre_resid[i][0]['val_rmse'] > pre_resid[j][0]['val_rmse']:
                    drop.add(i)
                    print(f'  peer corr {c:.4f} >= {THR_CORR_PEER} → drop {pre_resid[i][0]["cell_name"]} (val higher)')
                else:
                    drop.add(j)
                    print(f'  peer corr {c:.4f} >= {THR_CORR_PEER} → drop {pre_resid[j][0]["cell_name"]} (val higher)')
    final_passed = [pre_resid[i][0] for i in keep if i not in drop]
else:
    final_passed = pre_passed

# gate.csv 갱신
gate_df['final_pass'] = gate_df.apply(
    lambda row: any(r['trial']==row['trial'] and r['study']==row['study'] for r in final_passed),
    axis=1,
)
gate_df.to_csv(os.path.join(OUT_DIR, 'gate.csv'), index=False)

print(f'\n=== 최종 통과 후보 (3 게이트 모두): {len(final_passed)} ===')
for r in final_passed:
    print(f'  {r["study"]:25s} {r["cell_name"]:30s}  val={r["val_rmse"]:.6f}  test={r["test_rmse"]:.6f}')

전체 신규 trial: 9

=== 게이트 1+2 검사 결과 ===
              study  trial            cell_name  val_rmse  max_corr_vs_11           argmax_corr  pass_1  pass_2  pre_pass
B_two_stage_reverse      3  regression__top_200  0.005707        0.998774 bag_zit_combined_best    True   False     False
         A_reg_lgbm      2    tweedie_1.7__none  0.005745        0.998940             reg__lgbm    True   False     False
         A_reg_lgbm      3   tweedie_1.7__log1p  0.005745        0.998945             reg__lgbm    True   False     False
         A_reg_lgbm      1    tweedie_1.8__none  0.005750        0.998801             reg__lgbm    True   False     False
         A_reg_lgbm      0   tweedie_1.8__log1p  0.005750        0.998804             reg__lgbm    True   False     False
B_two_stage_reverse      5 tweedie_1.7__top_200  0.005795        0.990874 bag_zit_combined_best    True   False     False
B_two_stage_reverse      2    tweedie_1.7__full  0.005799        0.992056 bag_zit_combined_best    True   Fa

## 10. Stacking 비교

3가지 비교:
- **(a) 11-base only** (sanity check, val=0.005701 재현 확인)
- **(b) 11-base + 게이트 통과 모두**
- **(c) 11-base + 통과 후보 중 단독 val 낮은 top-3** (corr 다양성 우선 + val 우수만)

In [10]:
def _stack_with_enet_cv(X_oof, y_oof, X_val, y_val, X_test, y_test, model_names):
    """ElasticNetCV stacking. 비음수 제약 없음 (stacking_11base 정책 따름)."""
    enet = ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
        cv=5, random_state=SEED, n_jobs=-1, max_iter=10000,
    )
    enet.fit(X_oof, y_oof)
    oof_pred  = enet.predict(X_oof)
    val_pred  = enet.predict(X_val)
    test_pred = enet.predict(X_test)
    rmse_oof  = float(np.sqrt(np.mean((oof_pred  - y_oof)  ** 2)))
    rmse_val  = float(np.sqrt(np.mean((val_pred  - y_val)  ** 2)))
    rmse_test = float(np.sqrt(np.mean((test_pred - y_test) ** 2)))
    coefs = dict(zip(model_names, enet.coef_))
    return {
        'rmse_oof': rmse_oof, 'rmse_val': rmse_val, 'rmse_test': rmse_test,
        'alpha': float(enet.alpha_), 'l1_ratio': float(enet.l1_ratio_),
        'intercept': float(enet.intercept_), 'coefs': coefs,
    }


# (a) 11-base only
base_names_11 = list(BASE_OOF_PATHS.keys())
X_oof_11  = np.column_stack([base_oof_pred[k]  for k in base_names_11])
X_val_11  = np.column_stack([base_val_pred[k]  for k in base_names_11])
X_test_11 = np.column_stack([base_test_pred[k] for k in base_names_11])
y_oof_arr  = y_train_unit.values
y_val_arr  = y_val_unit.values
y_test_arr = y_test_unit.values

print('--- (a) 11-base only ---')
res_a = _stack_with_enet_cv(X_oof_11, y_oof_arr, X_val_11, y_val_arr, X_test_11, y_test_arr, base_names_11)
print(f'  oof={res_a["rmse_oof"]:.6f}  val={res_a["rmse_val"]:.6f}  test={res_a["rmse_test"]:.6f}')

results_compare = [
    {'method': '(a) 11-base only', 'oof': res_a['rmse_oof'], 'val': res_a['rmse_val'], 'test': res_a['rmse_test'], 'n_base': 10},
]

# (b) 11-base + 게이트 통과 모두
if len(final_passed) > 0:
    extra_names = [f'{r["study"]}__{r["cell_name"]}' for r in final_passed]
    X_oof_b  = np.column_stack([X_oof_11]  + [r['oof_pred']  for r in final_passed])
    X_val_b  = np.column_stack([X_val_11]  + [r['val_pred']  for r in final_passed])
    X_test_b = np.column_stack([X_test_11] + [r['test_pred'] for r in final_passed])
    print(f'\n--- (b) 11-base + {len(final_passed)} 통과 후보 ---')
    res_b = _stack_with_enet_cv(X_oof_b, y_oof_arr, X_val_b, y_val_arr, X_test_b, y_test_arr, base_names_11 + extra_names)
    print(f'  oof={res_b["rmse_oof"]:.6f}  val={res_b["rmse_val"]:.6f}  test={res_b["rmse_test"]:.6f}')
    results_compare.append({'method': f'(b) 11-base + {len(final_passed)} pass', 'oof': res_b['rmse_oof'], 'val': res_b['rmse_val'], 'test': res_b['rmse_test'], 'n_base': 10 + len(final_passed)})
else:
    print('\n(b) 통과 후보 0개 → 스킵')
    res_b = None

# (c) 11-base + 통과 후보 중 val 낮은 top-3
if len(final_passed) >= 1:
    top3 = sorted(final_passed, key=lambda r: r['val_rmse'])[:3]
    extra_names_c = [f'{r["study"]}__{r["cell_name"]}' for r in top3]
    X_oof_c  = np.column_stack([X_oof_11]  + [r['oof_pred']  for r in top3])
    X_val_c  = np.column_stack([X_val_11]  + [r['val_pred']  for r in top3])
    X_test_c = np.column_stack([X_test_11] + [r['test_pred'] for r in top3])
    print(f'\n--- (c) 11-base + top-{len(top3)} (val 낮은) ---')
    res_c = _stack_with_enet_cv(X_oof_c, y_oof_arr, X_val_c, y_val_arr, X_test_c, y_test_arr, base_names_11 + extra_names_c)
    print(f'  oof={res_c["rmse_oof"]:.6f}  val={res_c["rmse_val"]:.6f}  test={res_c["rmse_test"]:.6f}')
    results_compare.append({'method': f'(c) 11-base + top-{len(top3)}', 'oof': res_c['rmse_oof'], 'val': res_c['rmse_val'], 'test': res_c['rmse_test'], 'n_base': 10 + len(top3)})
else:
    res_c = None

cmp_df = pd.DataFrame(results_compare)
cmp_df.to_csv(os.path.join(OUT_DIR, 'stacking', 'comparison.csv'), index=False)
print(f'\n=== Stacking 비교 ===')
print(cmp_df.to_string(index=False))

# 베스트 stack의 coefs도 저장
best_stack = min([r for r in [res_a, res_b, res_c] if r is not None], key=lambda r: r['rmse_val'])
with open(os.path.join(OUT_DIR, 'stacking', 'best_stack_meta.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'rmse_oof':  best_stack['rmse_oof'],
        'rmse_val':  best_stack['rmse_val'],
        'rmse_test': best_stack['rmse_test'],
        'alpha':     best_stack['alpha'],
        'l1_ratio':  best_stack['l1_ratio'],
        'intercept': best_stack['intercept'],
        'coefs':     best_stack['coefs'],
    }, f, indent=2, ensure_ascii=False, default=str)

--- (a) 11-base only ---
  oof=0.005492  val=0.005702  test=0.008406

(b) 통과 후보 0개 → 스킵

=== Stacking 비교 ===
          method      oof      val     test  n_base
(a) 11-base only 0.005492 0.005702 0.008406      10


## 11. 결과 요약 + 결정

In [11]:
STACKING_11BASE_VAL = 0.005701   # 기존 plateau

print('=' * 80)
print(' last_test 실험 요약')
print('=' * 80)
print(f'  실행 trial: Study A {len(study_a_records)}개 + Study B {len(study_b_records)}개 = {len(all_records)}개')
print(f'  게이트 통과: {len(final_passed)}개')
print('-' * 80)
print(f'  비교 결과:')
for r in results_compare:
    delta = (r['val'] - STACKING_11BASE_VAL) * 1e6
    sign = '+' if delta > 0 else ''
    print(f'    {r["method"]:35s}  val={r["val"]:.6f}  test={r["test"]:.6f}  Δval(vs 11base)={sign}{delta:.1f}e-6')
print('-' * 80)

# 결정
best_method = min(results_compare, key=lambda r: r['val'])
if best_method['val'] < STACKING_11BASE_VAL - 1e-6:
    print(f'  ✅ 개선됨! best={best_method["method"]} val={best_method["val"]:.6f}')
    print(f'     → {OUT_DIR}/stacking/best_stack_meta.json 확인 후 final/로 이동 권장')
elif best_method['val'] < STACKING_11BASE_VAL + 1e-6:
    print(f'  ➖ 동일 수준 (val Δ < 1e-6). plateau 도달 확정.')
else:
    print(f'  ❌ 개선 안 됨. 11-base 그대로 유지 권장.')
    if not os.path.exists(os.path.join(OUT_DIR, 'CONCLUSION.md')):
        with open(os.path.join(OUT_DIR, 'CONCLUSION.md'), 'w', encoding='utf-8') as f:
            f.write(f'# last_test 결론\n\n')
            f.write(f'- best stacking val: {best_method["val"]:.6f}\n')
            f.write(f'- 11-base val: {STACKING_11BASE_VAL:.6f}\n')
            f.write(f'- 결과: loss/feature/target 다양화로 plateau 못 깸\n')
            f.write(f'- 통과 후보 {len(final_passed)}개에도 불구하고 stacking 개선 0\n')
            f.write(f'- information ceiling 도달 확정\n')
print('=' * 80)

# Colab → 로컬 자동 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip_base = os.path.join('/content', 'last_test_outputs')
    _zip_path = shutil.make_archive(_zip_base, 'zip', OUT_DIR)
    print(f'\n[zip 생성] {_zip_path} ({os.path.getsize(_zip_path)/1024/1024:.1f} MB)')
    try:
        files.download(_zip_path)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip_path))
except ImportError:
    pass

 last_test 실험 요약
  실행 trial: Study A 4개 + Study B 5개 = 9개
  게이트 통과: 0개
--------------------------------------------------------------------------------
  비교 결과:
    (a) 11-base only                     val=0.005702  test=0.008406  Δval(vs 11base)=+0.7e-6
--------------------------------------------------------------------------------
  ➖ 동일 수준 (val Δ < 1e-6). plateau 도달 확정.
